In [ ]:
from astropy.io import fits
import plotly.express as px
import plotly.colors as pc
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from generate_3d_mesh import spherical_mesh, visualize_mesh

In [ ]:
filename = '../data/sunerf_map.fits'

In [ ]:
with fits.open(filename) as hdul:
    hdul.info()
    data = np.flip(hdul[0].data, axis=0)

In [ ]:
fig = px.imshow(np.flip(data, axis=0))
fig.update_layout(
    template='plotly_dark',
    xaxis=dict(
        tickmode='array',
        tickvals=[60, 160, 260, 360, 460, 560, 660],    # 50 Carrington longitudes equals to 100 pixel. Centered at 360.
        ticktext=[-150, -100, -50, 0, 50, 100, 150]     # Carrington longitude
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=[20, 60, 100, 140, 180, 220, 260, 300, 340],   # 20 Carrington latitude equals 40 pixel. Centered at 180.
        ticktext=[80, 60, 40, 20, 0, -20, -40, -60, -80]        # Carrington latitude
    )
)
fig.show()

In [ ]:
longitudes = np.linspace(-180, 180, data.shape[1], endpoint=False)
latitudes = np.linspace(-90, 90, data.shape[0], endpoint=False)

# Create a meshgrid for all combinations
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# Flatten the grid
lons = lon_grid.flatten()
lats = lat_grid.flatten()

In [ ]:
# Create distances based on (normalized) data
base_radius = 10
data_normalized = (data - np.min(data)) / (np.max(data) - np.min(data))
distances = base_radius + data_normalized

In [ ]:
# Combine into points array
points = np.column_stack((lons, lats, distances.flatten()))

In [ ]:
# Generate mesh
vertices, triangles = spherical_mesh(points)

print(f"Mesh created with {len(vertices)} vertices and {len(triangles)} triangles")

In [ ]:
# Define a colormap and convert to it to one that Plotly can handle
cmap = plt.get_cmap('sdoaia304')
colorscale = pc.make_colorscale([matplotlib.colors.rgb2hex(cmap(i)) for i in np.linspace(0, 1, 10)])

In [ ]:
fig = visualize_mesh(vertices, triangles, "SDO Sphere", renderer='notebook', return_fig=True, edges=True, scatters=True)
fig.update_layout(template='plotly_dark', showlegend=False)
fig.update_traces(
    marker=dict(
        color=data_normalized.flatten(),
        colorscale=colorscale
    ),
    selector=dict(mode='markers')
)
fig.show(renderer='browser')

In [ ]:
#export_mesh(vertices, triangles, "output/sdo_test", "stl")